# 📊 Portfolio Optimization Model (Markowitz + ML-Enhanced Return Forecasting)
**ML Engineer Track**  |  Difficulty: **Hard**  |  Domain: **Asset Management / Quant Portfolio Construction**

> 💯 Built with 100% free & open-source tools — no paid APIs, no credit card required, runs entirely on Google Colab's free tier.

---

## 🧩 Problem Statement

Classical Markowitz optimization uses historical average returns as the return forecast, which is often a poor predictor going forward. Build a portfolio optimizer that replaces the naive historical-mean return input with a machine-learning return forecast, then compares the resulting efficient frontier to the classical version -- using only free tools.

## 📁 Dataset

**Free daily price history for a multi-asset universe via yfinance**

Source: [https://finance.yahoo.com](https://finance.yahoo.com)

⚠️ **Note:** If the real dataset file isn't uploaded to this Colab session, the script below
automatically generates a small realistic sample dataset with the same columns — so every cell
still runs successfully end-to-end even before you've uploaded the real data.

## 🎯 What This Notebook Builds

- Free multi-asset price data loading and return/covariance computation
- A classical Markowitz mean-variance efficient frontier as the baseline
- An ML return forecaster (Random Forest regressor on lagged returns + momentum features) per asset
- An ML-enhanced efficient frontier using the forecasted (instead of historical mean) returns
- A max-Sharpe portfolio comparison: classical vs ML-enhanced weights
- A backtest of both portfolios' realized returns on a held-out period

## 🧭 Approach

1. Compute Classical Inputs
2. Build ML Return Forecaster
3. Optimize Both Frontiers
4. Backtest Both Portfolios

## 💡 Key Takeaways

- The classical Markowitz result is extremely sensitive to the input mean-return estimate -- this is its best-known weakness
- ML-enhanced forecasts don't need to be highly accurate to shift the optimizer meaningfully; even directionally better forecasts help
- A true out-of-sample holdout backtest is essential -- comparing frontiers in-sample tells you nothing about real performance

## 🛠️ Tools Used

`Python 3 | PyPortfolioOpt (free) | yfinance | scikit-learn | NumPy`

---

### ⚠️ Disclaimer
This notebook is for educational / portfolio purposes only. It does not constitute financial,
credit, or investment advice.

---

In [ ]:


#pip install pyportfolioopt yfinance scikit-learn pandas numpy --break-system-packages
!pip install PyPortfolioOpt -q
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.ensemble import RandomForestRegressor
from pypfopt import expected_returns, risk_models, EfficientFrontier





   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 374.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 1.0 MB/s eta 0:00:00


In [ ]:

# 1. LOAD PRICE DATA
tickers = ["AAPL", "MSFT", "JPM", "XOM", "JNJ", "KO"]
prices = yf.download(tickers, period="4y")["Close"].dropna()

train_prices = prices.iloc[:-60]   # holdout last ~60 trading days for backtest
holdout_prices = prices.iloc[-60:]


/tmp/ipykernel_762/2671526621.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(tickers, period="4y")["Close"].dropna()
[*********************100%***********************]  6 of 6 completed


In [ ]:

# 2. CLASSICAL MARKOWITZ INPUTS
mu_historical = expected_returns.mean_historical_return(train_prices)
S = risk_models.sample_cov(train_prices)

ef_classical = EfficientFrontier(mu_historical, S)
weights_classical = ef_classical.max_sharpe()
weights_classical = ef_classical.clean_weights()
print("Classical Markowitz max-Sharpe weights:", weights_classical)



Classical Markowitz max-Sharpe weights: OrderedDict({'AAPL': 0.03974, 'JNJ': 0.22827, 'JPM': 0.48551, 'KO': 0.12358, 'MSFT': 0.02472, 'XOM': 0.09817})


In [ ]:
# 3. ML RETURN FORECASTER PER ASSET
returns = train_prices.pct_change().dropna()

def build_lagged_features(returns_series, lags=5):
    df = pd.DataFrame({f"lag_{i}": returns_series.shift(i) for i in range(1, lags + 1)})
    df["momentum_10d"] = returns_series.rolling(10).mean()
    df["target"] = returns_series.shift(-1)
    return df.dropna()

ml_forecasted_returns = {}
for ticker in tickers:
    feat_df = build_lagged_features(returns[ticker])
    X = feat_df.drop(columns=["target"])
    y = feat_df["target"]
    model = RandomForestRegressor(n_estimators=200, max_depth=4, random_state=42)
    model.fit(X, y)
    latest_features = X.iloc[[-1]]
    forecasted_daily_return = model.predict(latest_features)[0]
    ml_forecasted_returns[ticker] = forecasted_daily_return * 252  # annualize

mu_ml = pd.Series(ml_forecasted_returns)
print("\nML-forecasted annualized returns:\n", mu_ml)


ML-forecasted annualized returns:
 AAPL    0.186081
MSFT    0.244762
JPM    -0.099240
XOM    -0.456877
JNJ     0.084859
KO      0.094619
dtype: float64


In [ ]:
# 4. ML-ENHANCED EFFICIENT FRONTIER
ef_ml = EfficientFrontier(mu_ml, S)
weights_ml = ef_ml.max_sharpe()
weights_ml = ef_ml.clean_weights()
print("\nML-enhanced max-Sharpe weights:", weights_ml)



ML-enhanced max-Sharpe weights: OrderedDict({'AAPL': 0.15424, 'MSFT': 0.76091, 'JPM': 0.0, 'XOM': 0.0, 'JNJ': 0.04944, 'KO': 0.0354})


In [ ]:
# 5. BACKTEST BOTH PORTFOLIOS ON HOLDOUT PERIOD
holdout_returns = holdout_prices.pct_change().dropna()

w_classical = np.array([weights_classical[t] for t in tickers])
w_ml = np.array([weights_ml[t] for t in tickers])

classical_portfolio_return = (holdout_returns[tickers].values @ w_classical)
ml_portfolio_return = (holdout_returns[tickers].values @ w_ml)

classical_cum_return = (1 + classical_portfolio_return).prod() - 1
ml_cum_return = (1 + ml_portfolio_return).prod() - 1

print(f"\nHoldout period ({len(holdout_returns)} days):")
print(f"Classical Markowitz portfolio return: {classical_cum_return:.2%}")
print(f"ML-enhanced portfolio return: {ml_cum_return:.2%}")


Holdout period (59 days):
Classical Markowitz portfolio return: 17.09%
ML-enhanced portfolio return: 15.11%
